# Mech Int

Idea: Does LLMs have sense of trade-offs?
* Replicate results grom Cash or Comfort paper with small model
* SFT on the responses that are not 'aligned' with expectations
* See if we can improve the model's performance on the task
* Investigate if retrained model can transfer 'learned' representations of the trade-offs to different prompts


# analysis

In [1]:
import torch
import gc
import warnings
import os

# Suppress transformers warnings
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

# Suppress bitsandbytes warnings (optional)
warnings.filterwarnings('ignore', category=UserWarning, module='bitsandbytes')

torch.cuda.empty_cache()
gc.collect()

20

In [2]:
import functools
import sys
from pathlib import Path
from typing import Callable

import circuitsvis as cv
import einops
import numpy as np
import torch as t
import torch.nn as nn
import torch.nn.functional as F

from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor
from tqdm import tqdm
from transformer_lens import (
    ActivationCache,
    FactoredMatrix,
    HookedTransformer,
    HookedTransformerConfig,
    utils,
)
from neel_plotly import *

model: HookedTransformer = HookedTransformer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", device='cuda')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model meta-llama/Llama-3.2-3B-Instruct into HookedTransformer


In [3]:
df = pd.read_csv('/teamspace/studios/this_studio/MATS/data/llama3.2-3bI_hunger_21x21.csv')
df.head()

,reward_euros,quant_minutes,answer,prompt
0,0.1,0,Yes,\nAnswer with one word. Do you accept compensa...
1,0.1,30,Ja,\nAnswer with one word. Do you accept compensa...
2,0.1,60,Ja,\nAnswer with one word. Do you accept compensa...
3,0.1,90,Ja,\nAnswer with one word. Do you accept compensa...
4,0.1,120,Yes,\nAnswer with one word. Do you accept compensa...


In [4]:
df[df['quant_minutes']==330]
# df.iloc[305]['prompt']

,reward_euros,quant_minutes,answer,prompt
11,0.10,330,Yes,\nAnswer with one word. Do you accept compensa...
32,0.16,330,Yes,\nAnswer with one word. Do you accept compensa...
53,0.25,330,Yes,\nAnswer with one word. Do you accept compensa...
74,0.40,330,Yes,\nAnswer with one word. Do you accept compensa...
95,0.63,330,Yes,\nAnswer with one word. Do you accept compensa...
116,1.00,330,Si,\nAnswer with one word. Do you accept compensa...
137,1.58,330,Yes,\nAnswer with one word. Do you accept compensa...
158,2.51,330,Yes,\nAnswer with one word. Do you accept compensa...
179,3.98,330,Yes,\nAnswer with one word. Do you accept compensa...
200,6.31,330,Yes,\nAnswer with one word. Do you accept compensa...


In [5]:
df.iloc[431]['prompt']

'\nAnswer with one word. Do you accept compensation of 1000.00 euros for waiting 330 minutes for food?\n'

In [6]:
prompt = """
Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?
"""
# prompt_yesish = """
# Answer with one word. Do you accept compensation of 10.00 euros for waiting 360 minutes for food?
# """

In [7]:
from transformer_lens.utils import test_prompt

# Test the model with a prompt
test_prompt(
    prompt,
    "No",
    model,
    prepend_space_to_answer=False,
)

# test_prompt(
#     prompt_yesish,
#     "Yes",
#     model,
#     prepend_space_to_answer=False,
# )

Tokenized prompt: ['<|begin_of_text|>', '\n', 'Answer', ' with', ' one', ' word', '.', ' Do', ' you', ' accept', ' compensation', ' of', ' ', '100', '.', '00', ' euros', ' for', ' waiting', ' ', '330', ' minutes', ' for', ' food', '?\n']
Tokenized answer: ['No']


Performance on answer token:
Rank: 0        Logit: 14.73 Prob: 21.29% Token: |No|

Top 0th token. Logit: 14.73 Prob: 21.29% Token: |No|
Top 1th token. Logit: 14.57 Prob: 18.11% Token: |Yes|
Top 2th token. Logit: 14.27 Prob: 13.45% Token: |Si|
Top 3th token. Logit: 12.89 Prob:  3.36% Token: |Ja|
Top 4th token. Logit: 12.62 Prob:  2.57% Token: |S|
Top 5th token. Logit: 12.22 Prob:  1.73% Token: |Ne|
Top 6th token. Logit: 12.22 Prob:  1.73% Token: |no|
Top 7th token. Logit: 11.96 Prob:  1.34% Token: | Si|
Top 8th token. Logit: 11.90 Prob:  1.25% Token: |Do|
Top 9th token. Logit: 11.51 Prob:  0.85% Token: | Ja|


Ranks of the answer tokens: [('No', 0)]

In [8]:
# 63.10 euros & 330 mins

# forward pass with cache
logits, cache = model.run_with_cache(prompt)

# convert to tokens
tokens = model.to_str_tokens(prompt)
print("Tokens:", tokens)
print("Logits shape:", logits.shape)

Tokens: ['<|begin_of_text|>', '\n', 'Answer', ' with', ' one', ' word', '.', ' Do', ' you', ' accept', ' compensation', ' of', ' ', '100', '.', '00', ' euros', ' for', ' waiting', ' ', '330', ' minutes', ' for', ' food', '?\n']
Logits shape: torch.Size([1, 25, 128256])


In [9]:
# 2. look at next-token distribution
# take logits at the last position
last_logits = logits[0, -1, :]  # shape [vocab]
probs = torch.softmax(last_logits, dim=-1)

# check probabilities for Yes/No
yes_id = model.to_single_token("Yes")
no_id  = model.to_single_token("No")

print("P(Yes):", probs[yes_id].item())
print("P(No): ", probs[no_id].item())

# top 5 most probable next tokens
topk = torch.topk(probs, k=5)
top_ids = topk.indices.tolist()
top_probs = topk.values.tolist()

print("\nTop 5 next tokens:")
for tid, p in zip(top_ids, top_probs):
    token_str = model.to_string(tid)
    print(f"{token_str!r} : {p:.4f}")


P(Yes): 0.18107393383979797
P(No):  0.2129419445991516

Top 5 next tokens:
'No' : 0.2129
'Yes' : 0.1811
'Si' : 0.1345
'Ja' : 0.0336
'S' : 0.0257


In [10]:
tokens = model.to_tokens(prompt)


# Run model with cache
logits, cache = model.run_with_cache(tokens)

# Get Yes and No tokens
yes_token = model.to_single_token("Yes")
no_token = model.to_single_token("No")

# Get residual stream decomposition
decomposed_resid, labels = cache.get_full_resid_decomposition(
    expand_neurons=False, 
    pos_slice=-1,  # Last position
    return_labels=True
)

# Apply layer norm
decomposed_resid = cache.apply_ln_to_stack(decomposed_resid, pos_slice=-1)
print(f"Decomposed residual shape: {decomposed_resid.shape}")

# Get unembedding vectors for Yes and No
yes_unembed = model.W_U[:, yes_token]
no_unembed = model.W_U[:, no_token]

# Calculate logit difference (Yes - No)
logit_diff = yes_unembed - no_unembed

# Calculate direct logit attribution for each component
dla = einops.einsum(
    decomposed_resid, 
    logit_diff, 
    "component batch d_model, d_model -> component batch"
)

# Since we have only one prompt, we can squeeze the batch dimension
dla_single = dla[:, 0]  # Shape: [component]

# Plot the results
line(
    dla_single, 
    x=labels, 
    title="Direct Logit Attribution: Yes vs No", 
    xaxis="Component",
    yaxis="Logit Difference (Yes - No)"
)

# Since we have only one prompt, we can squeeze the batch dimension
dla_single = dla[:, 0]  # Shape: [component]

# Filter to only show layer 24 onwards
layer_24_start_idx = None
for i, label in enumerate(labels):
    if "L23" in label:
        layer_24_start_idx = i
        break

if layer_24_start_idx is not None:
    dla_filtered = dla_single[layer_24_start_idx:]
    labels_filtered = labels[layer_24_start_idx:]
    
    # Plot the results (layer 24 onwards only)
    line(
        dla_filtered, 
        x=labels_filtered, 
        title="Direct Logit Attribution: Logit Difference (Yes - No) (Layer 24+)", 
        xaxis="Component",
        yaxis="Logit Difference (Yes - No)"
    )
    
    print(f"Showing {len(dla_filtered)} components from layer 24 onwards")
else:
    # Fallback to show all if layer 24 not found
    line(
        dla_single, 
        x=labels, 
        title="Direct Logit Attribution: Yes vs No", 
        xaxis="Component",
        yaxis="Logit Difference (Yes - No)"
    )
    print("Layer 24 not found, showing all components")

# Optional: Show the actual logit values for Yes and No
final_logits = logits[0, -1, :]  # Last position, single batch
yes_logit = final_logits[yes_token].item()
no_logit = final_logits[no_token].item()
print(f"Final Yes logit: {yes_logit:.3f}")
print(f"Final No logit: {no_logit:.3f}")
print(f"Logit difference (Yes - No): {yes_logit - no_logit:.3f}")
print(f"Sum of DLA components: {dla_single.sum().item():.3f}")

# Optional: Show top contributing components from layer 24 onwards
if layer_24_start_idx is not None:
    top_k = 10
    top_indices = torch.topk(dla_filtered, k=min(top_k, len(dla_filtered))).indices
    print(f"\nTop {min(top_k, len(dla_filtered))} components favoring Yes (Layer 23+):")
    for i, idx in enumerate(top_indices):
        print(f"{i+1}. {labels_filtered[idx]}: {dla_filtered[idx].item():.3f}")

    bottom_indices = torch.topk(dla_filtered, k=min(top_k, len(dla_filtered)), largest=False).indices
    print(f"\nTop {min(top_k, len(dla_filtered))} components favoring No (Layer 23+):")
    for i, idx in enumerate(bottom_indices):
        print(f"{i+1}. {labels_filtered[idx]}: {dla_filtered[idx].item():.3f}")
else:
    top_k = 10
    top_indices = torch.topk(dla_single, k=top_k).indices
    print(f"\nTop {top_k} components favoring Yes:")
    for i, idx in enumerate(top_indices):
        print(f"{i+1}. {labels[idx]}: {dla_single[idx].item():.3f}")

    bottom_indices = torch.topk(dla_single, k=top_k, largest=False).indices
    print(f"\nTop {top_k} components favoring No:")
    for i, idx in enumerate(bottom_indices):
        print(f"{i+1}. {labels[idx]}: {dla_single[idx].item():.3f}")

Tried to stack head results when they weren't cached. Computing head results now
Decomposed residual shape: torch.Size([702, 1, 3072])


Showing 150 components from layer 24 onwards
Final Yes logit: 14.569
Final No logit: 14.731
Logit difference (Yes - No): -0.162
Sum of DLA components: -0.162

Top 10 components favoring Yes (Layer 23+):
1. 19_mlp_out: 4.853
2. 22_mlp_out: 3.422
3. 15_mlp_out: 0.543
4. L26H0: 0.511
5. 14_mlp_out: 0.498
6. L23H17: 0.429
7. L23H13: 0.350
8. L27H11: 0.321
9. 17_mlp_out: 0.316
10. L27H7: 0.279

Top 10 components favoring No (Layer 23+):
1. 21_mlp_out: -7.487
2. 25_mlp_out: -3.110
3. 26_mlp_out: -0.601
4. 18_mlp_out: -0.418
5. L26H1: -0.404
6. 24_mlp_out: -0.380
7. L26H2: -0.287
8. L27H2: -0.286
9. 8_mlp_out: -0.189
10. 11_mlp_out: -0.134


In [11]:
# Optional: Show the actual logit values for Yes and No
final_logits = logits[0, -1, :]  # Last position, single batch
yes_logit = final_logits[yes_token].item()
no_logit = final_logits[no_token].item()
print(f"Final Yes logit: {yes_logit:.3f}")
print(f"Final No logit: {no_logit:.3f}")
print(f"Logit difference (Yes - No): {yes_logit - no_logit:.3f}")
print(f"Sum of DLA components: {dla_single.sum().item():.3f}")

# Plot Yes vs No probabilities across layers
yes_probs, no_probs = [], []
layer_numbers = []

for layer in range(model.cfg.n_layers):
    hidden = cache[f"blocks.{layer}.hook_resid_post"][0, -1, :]
    logits_layer = model.unembed(model.ln_final(hidden))
    probs_layer = torch.softmax(logits_layer, dim=-1)
    yes_probs.append(probs_layer[yes_token].item())
    no_probs.append(probs_layer[no_token].item())
    layer_numbers.append(layer)

# Create probability plot using neel_plotly
import numpy as np
prob_data = np.stack([yes_probs, no_probs], axis=0)  # Shape: [2, n_layers]

line(
    prob_data, 
    x=layer_numbers,
    title="Yes vs No Probabilities Across Layers (€1,000, 330 mins)",
    xaxis="Layer", 
    yaxis="Probability",
    line_labels=["Yes", "No"]
)

Final Yes logit: 14.569
Final No logit: 14.731
Logit difference (Yes - No): -0.162
Sum of DLA components: -0.162


In [12]:
prompt

'\nAnswer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?\n'

In [13]:
# Optional: Show the actual logit values for Yes and No
final_logits = logits[0, -1, :]  # Last position, single batch
yes_logit = final_logits[yes_token].item()
no_logit = final_logits[no_token].item()
print(f"Final Yes logit: {yes_logit:.3f}")
print(f"Final No logit: {no_logit:.3f}")
print(f"Logit difference (Yes - No): {yes_logit - no_logit:.3f}")
print(f"Sum of DLA components: {dla_single.sum().item():.3f}")

# Plot Yes vs No probabilities across layers
yes_probs, no_probs = [], []
layer_numbers = []

for layer in range(model.cfg.n_layers):
    hidden = cache[f"blocks.{layer}.hook_resid_post"][0, -1, :]
    logits_layer = model.unembed(model.ln_final(hidden))
    probs_layer = torch.softmax(logits_layer, dim=-1)
    yes_probs.append(probs_layer[yes_token].item())
    no_probs.append(probs_layer[no_token].item())
    layer_numbers.append(layer)

# Create probability plot using neel_plotly
import numpy as np
prob_data = np.stack([yes_probs, no_probs], axis=0)  # Shape: [2, n_layers]

line(
    prob_data, 
    x=layer_numbers,
    title="Yes vs No Probabilities Across Layers (€1,000, 330 mins)",
    xaxis="Layer", 
    yaxis="Probability",
    line_labels=["Yes", "No"]
)

Final Yes logit: 14.569
Final No logit: 14.731
Logit difference (Yes - No): -0.162
Sum of DLA components: -0.162


In [14]:
# Plot the results
line(
    dla_single, 
    x=labels, 
    title="Direct Logit Attribution: Logit Difference (Yes - No)", 
    xaxis="Component",
    yaxis="Logit Difference (Yes - No)"
)

In [15]:
# Get residual stream decomposition
decomposed_resid, labels = cache.get_full_resid_decomposition(
    expand_neurons=False, 
    pos_slice=-1,  # Last position
    return_labels=True
)

# Apply layer norm
decomposed_resid = cache.apply_ln_to_stack(decomposed_resid, pos_slice=-1)
print(f"Decomposed residual shape: {decomposed_resid.shape}")

# Get unembedding vectors for Yes and No separately
yes_unembed = model.W_U[:, yes_token]
no_unembed = model.W_U[:, no_token]

# Calculate direct logit attribution for each component - SEPARATELY for each token
yes_dla = einops.einsum(
    decomposed_resid, 
    yes_unembed, 
    "component batch d_model, d_model -> component batch"
)

no_dla = einops.einsum(
    decomposed_resid, 
    no_unembed, 
    "component batch d_model, d_model -> component batch"
)

# Since we have only one prompt, squeeze the batch dimension
yes_dla_single = yes_dla[:, 0]  # Shape: [component]
no_dla_single = no_dla[:, 0]    # Shape: [component]

# Filter to only show layer 24 onwards
layer_24_start_idx = None
for i, label in enumerate(labels):
    if "L25" in label:
        layer_24_start_idx = i
        break

if layer_24_start_idx is not None:
    yes_dla_filtered = yes_dla_single[layer_24_start_idx:]
    no_dla_filtered = no_dla_single[layer_24_start_idx:]
    labels_filtered = labels[layer_24_start_idx:]
    
    # Plot both logit attributions on the same plot
    attributions_combined = t.stack([yes_dla_filtered, no_dla_filtered], dim=0)  # Shape: [2, components]
    
    line(
        attributions_combined, 
        x=labels_filtered, 
        title="Direct Logit Attribution: Yes and No Tokens (Layer 25+) (€1,000, 330 mins)", 
        xaxis="Component",
        yaxis="Logit Attribution",
        line_labels=["Yes", "No"]
    )
    
    print(f"Showing {len(yes_dla_filtered)} components from layer 27 onwards")
else:
    # Fallback to show all if layer 24 not found
    attributions_combined = t.stack([yes_dla_single, no_dla_single], dim=0)
    
    line(
        attributions_combined, 
        x=labels, 
        title="Direct Logit Attribution: Yes and No Tokens", 
        xaxis="Component",
        yaxis="Logit Attribution",
        line_labels=["Yes", "No"]
    )
    print("Layer 24 not found, showing all components")

# Optional: Show the actual logit values for Yes and No
final_logits = logits[0, -1, :]  # Last position, single batch
yes_logit = final_logits[yes_token].item()
no_logit = final_logits[no_token].item()
print(f"Final Yes logit: {yes_logit:.3f}")
print(f"Final No logit: {no_logit:.3f}")
print(f"Logit difference (Yes - No): {yes_logit - no_logit:.3f}")
print(f"Sum of Yes DLA components: {yes_dla_single.sum().item():.3f}")
print(f"Sum of No DLA components: {no_dla_single.sum().item():.3f}")

# Optional: Show top contributing components from layer 24 onwards
# if layer_24_start_idx is not None:
#     top_k = 3
    
#     # Top components for Yes
#     yes_top_indices = t.topk(yes_dla_filtered, k=min(top_k, len(yes_dla_filtered))).indices
#     print(f"\nTop {min(top_k, len(yes_dla_filtered))} components contributing to Yes:")
#     for i, idx in enumerate(yes_top_indices):
#         print(f"{i+1}. {labels_filtered[idx]}: {yes_dla_filtered[idx].item():.3f}")
    
#     # Top components for No
#     no_top_indices = t.topk(no_dla_filtered, k=min(top_k, len(no_dla_filtered))).indices
#     print(f"\nTop {min(top_k, len(no_dla_filtered))} components contributing to No:")
#     for i, idx in enumerate(no_top_indices):
#         print(f"{i+1}. {labels_filtered[idx]}: {no_dla_filtered[idx].item():.3f}")

top_k = 3

# Top components for Yes
yes_top_indices = t.topk(yes_dla_single, k=top_k).indices
print(f"\nTop {top_k} components contributing to 'Yes' (logits):")
for i, idx in enumerate(yes_top_indices):
    print(f"{i+1}. {labels[idx]}: {yes_dla_single[idx].item():.3f}")

# Top components for No
no_top_indices = t.topk(no_dla_single, k=top_k).indices
print(f"\nTop {top_k} components contributing to 'No' (logits):")
for i, idx in enumerate(no_top_indices):
    print(f"{i+1}. {labels[idx]}: {no_dla_single[idx].item():.3f}")

Decomposed residual shape: torch.Size([702, 1, 3072])


Showing 102 components from layer 27 onwards
Final Yes logit: 14.569
Final No logit: 14.731
Logit difference (Yes - No): -0.162
Sum of Yes DLA components: 14.569
Sum of No DLA components: 14.731

Top 3 components contributing to 'Yes' (logits):
1. 19_mlp_out: 6.159
2. 27_mlp_out: 3.394
3. 15_mlp_out: 0.967

Top 3 components contributing to 'No' (logits):
1. 21_mlp_out: 6.125
2. 27_mlp_out: 3.379
3. 25_mlp_out: 1.985


In [16]:
import torch
import torch.nn.functional as F
import numpy as np
import plotly.express as px

# Assumes you already ran:
# logits, cache = model.run_with_cache(prompt)

device = logits.device
n_layers = model.cfg.n_layers
n_heads  = model.cfg.n_heads

# Get token ids for "Yes"/"No" (try both w/ and w/o leading space)
def _tok_id(s):
    try:
        tid = model.to_single_token(s)
        return tid
    except Exception:
        return None

yes_id = _tok_id("Yes")
no_id  = _tok_id("No")
assert yes_id is not None and no_id is not None, "Couldn't find Yes/No token IDs."

# Unembed direction for Yes vs No: [d_model]
W_U = model.W_U.to(device)                       # [d_model, vocab]
yes_no_dir = W_U[:, yes_id] - W_U[:, no_id]      # [d_model]

# Heatmap array: layers x heads
head_attr = torch.zeros((n_layers, n_heads), device=device)

with torch.no_grad():
    for L in range(n_layers):
        # Head outputs at last token BEFORE W_O: hook_z → [batch, seq, n_heads, d_head]
        # We want batch=0, last position
        z = cache[f"blocks.{L}.attn.hook_z"][0, -1, :, :]            # [n_heads, d_head]

        # Per-head output projection matrices W_O[L]: [n_heads, d_head, d_model]
        W_O_L = model.W_O[L].to(device)

        # Map each head's z through its W_O to get residual contribution: [n_heads, d_model]
        # contrib[h] = z[h] @ W_O_L[h]
        contrib = torch.einsum("hd,hdm->hm", z, W_O_L)               # [n_heads, d_model]

        # Logit attribution for Yes-No: (contrib · (W_U_yes - W_U_no)) per head → [n_heads]
        head_attr[L] = contrib @ yes_no_dir                           # [n_heads]

# To CPU / numpy for Plotlya
attr_np = head_attr.detach().float().cpu().numpy()

# Plot as heatmap (layers × heads)
fig = px.imshow(
    attr_np,
    x=[f"H{h}" for h in range(n_heads)],
    y=[f"L{l}" for l in range(n_layers)],
    color_continuous_scale=px.colors.diverging.RdBu,
    color_continuous_midpoint=0.0,
    labels=dict(x="Attention head", y="Layer", color="Logit attribution diff."),
    title="Logit attribution difference (Yes-No) (€10, 330 mins)",
)

fig.show()


In [17]:
# import torch
# import torch.nn.functional as F

# def mean_ablate_head(model, prompt, L, H, yes_id=None, no_id=None, mode="last"):
#     """
#     Replace head (L,H) activations with their mean value (across sequence positions).
#     mode = "last" -> patch only the last token.
#     mode = "all"  -> patch all positions in the sequence.
#     Returns baseline and mean-ablated P(Yes)/P(No).
#     """
#     # resolve token IDs
#     def _id(s):
#         try: return model.to_single_token(s)
#         except Exception: return None
#     if yes_id is None or no_id is None:
#         yes_id = _id("Yes")
#         no_id  = _id("No")
#     assert yes_id is not None and no_id is not None

#     # baseline
#     logits_base, cache_base = model.run_with_cache(prompt)
#     last_logits = logits_base[0, -1, :]
#     probs_base = F.softmax(last_logits, dim=-1)
#     p_yes_base, p_no_base = probs_base[yes_id].item(), probs_base[no_id].item()

#     # compute mean z for this head across the sequence
#     z_all = cache_base[f"blocks.{L}.attn.hook_z"][0, :, H, :]   # [seq, d_head]
#     z_mean = z_all.mean(dim=0, keepdim=True)                    # [1, d_head]

#     # patch hook
#     def patch_to_mean(z, hook):
#         # z: [batch, seq, n_heads, d_head]
#         if mode == "last":
#             z[:, -1, H, :] = z_mean.to(z.dtype).to(z.device)
#         elif mode == "all":
#             z[:, :, H, :] = z_mean.to(z.dtype).to(z.device)
#         return z

#     with model.hooks([(f"blocks.{L}.attn.hook_z", patch_to_mean)]):
#         logits_ablate, _ = model.run_with_cache(prompt)

#     last_logits_ablate = logits_ablate[0, -1, :]
#     probs_ablate = F.softmax(last_logits_ablate, dim=-1)
#     p_yes_ablate, p_no_ablate = probs_ablate[yes_id].item(), probs_ablate[no_id].item()

#     print(f"[Mean-ablation L{L}H{H}]")
#     print(f"Baseline  P(Yes)={p_yes_base:.4f}, P(No)={p_no_base:.4f}")
#     print(f"Mean-abl  P(Yes)={p_yes_ablate:.4f}, P(No)={p_no_ablate:.4f}")
#     print(f"ΔYes={p_yes_base-p_yes_ablate:+.4f}, ΔNo={p_no_base-p_no_ablate:+.4f}")

#     return dict(
#         p_yes_base=p_yes_base, p_no_base=p_no_base,
#         p_yes_ablate=p_yes_ablate, p_no_ablate=p_no_ablate,
#         delta_yes=p_yes_base - p_yes_ablate,
#         delta_no=p_no_base  - p_no_ablate
#     )

# # Example: mean-ablate Layer 26, Head 1
# res = mean_ablate_head(model, prompt, L=26, H=10, mode="last")


In [18]:
# # Example: mean-ablate Layer 26, Head 1
# res = mean_ablate_head(model, prompt, L=27, H=2, mode="last")

In [19]:
import torch
import numpy as np
from transformer_lens import HookedTransformer
from neel_plotly import *


tokens = model.to_tokens(prompt)

# Get baseline (clean) run
clean_logits, clean_cache = model.run_with_cache(tokens)
yes_token = model.to_single_token("Yes")
no_token = model.to_single_token("No")

# Get baseline probabilities
clean_probs = torch.softmax(clean_logits[0, -1, :], dim=-1)
baseline_yes_prob = clean_probs[yes_token].item()
baseline_no_prob = clean_probs[no_token].item()
baseline_logit_diff = (clean_logits[0, -1, yes_token] - clean_logits[0, -1, no_token]).item()

print(f"Baseline - Yes: {baseline_yes_prob:.3f}, No: {baseline_no_prob:.3f}")
print(f"Baseline logit difference (Yes - No): {baseline_logit_diff:.3f}")

# Function to create mean ablation hook for a specific MLP layer
def create_mlp_ablation_hook(layer_idx, clean_cache):
    """Creates a hook that replaces MLP output with its mean across sequence positions"""
    def ablation_hook(mlp_out, hook):
        # mlp_out shape: [batch, seq_len, d_model]
        # Replace with mean across sequence dimension
        mean_activation = clean_cache[f"blocks.{layer_idx}.mlp.hook_post"].mean(dim=1, keepdim=True)
        # Broadcast to all sequence positions
        mlp_out[:] = mean_activation
        return mlp_out
    return ablation_hook

# Test ablation of individual MLP layers
n_layers = model.cfg.n_layers
layer_results = []
layers_to_test = range(max(0, n_layers-15), n_layers)  # Test last 15 layers

print(f"\nTesting ablation of MLP layers {min(layers_to_test)} to {max(layers_to_test)}:")
print("-" * 60)

for layer_idx in layers_to_test:
    # Create the ablation hook
    hook_fn = create_mlp_ablation_hook(layer_idx, clean_cache)
    
    # Run with ablation
    with model.hooks(fwd_hooks=[(f"blocks.{layer_idx}.mlp.hook_post", hook_fn)]):
        ablated_logits = model(tokens)
    
    # Calculate results
    ablated_probs = torch.softmax(ablated_logits[0, -1, :], dim=-1)
    ablated_yes_prob = ablated_probs[yes_token].item()
    ablated_no_prob = ablated_probs[no_token].item()
    ablated_logit_diff = (ablated_logits[0, -1, yes_token] - ablated_logits[0, -1, no_token]).item()
    
    # Calculate changes
    yes_prob_change = ablated_yes_prob - baseline_yes_prob
    no_prob_change = ablated_no_prob - baseline_no_prob
    logit_diff_change = ablated_logit_diff - baseline_logit_diff
    
    layer_results.append({
        'layer': layer_idx,
        'yes_prob_change': yes_prob_change,
        'no_prob_change': no_prob_change,
        'logit_diff_change': logit_diff_change,
        'ablated_yes_prob': ablated_yes_prob,
        'ablated_no_prob': ablated_no_prob
    })
    
    print(f"Layer {layer_idx:2d}: Yes Δ={yes_prob_change:+.3f}, No Δ={no_prob_change:+.3f}, "
          f"LogitDiff Δ={logit_diff_change:+.3f}")

# Convert to arrays for plotting
layers = [r['layer'] for r in layer_results]
yes_changes = [r['yes_prob_change'] for r in layer_results]
no_changes = [r['no_prob_change'] for r in layer_results]
logit_diff_changes = [r['logit_diff_change'] for r in layer_results]

# Plot probability changes
prob_changes = np.array([yes_changes, no_changes])
line(
    prob_changes,
    x=layers,
    title="Probability Changes from MLP Mean Ablation",
    xaxis="Layer",
    yaxis="Probability Change",
    line_labels=["Yes Change", "No Change"]
)

# Plot logit difference changes
line(
    logit_diff_changes,
    x=layers,
    title="Logit Difference Changes from MLP Mean Ablation",
    xaxis="Layer", 
    yaxis="Logit Difference Change (Yes - No)",
    line_labels=["Logit Diff Change"]
)

# Find most important layers
most_impactful = sorted(layer_results, key=lambda x: abs(x['logit_diff_change']), reverse=True)
print(f"\nMost impactful MLP layers (by absolute logit difference change):")
print("-" * 60)
for i, result in enumerate(most_impactful[:5]):
    print(f"{i+1}. Layer {result['layer']}: Logit diff change = {result['logit_diff_change']:+.3f}")
    print(f"   Final probs after ablation - Yes: {result['ablated_yes_prob']:.3f}, No: {result['ablated_no_prob']:.3f}")

# Test multiple layer ablation for the most impactful layers
#print(f"\nTesting simultaneous ablation of top 1 most impactful layer:")
#top_3_layers = [r['layer'] for r in most_impactful[:5]]

top_3_layers = [21]
print(f"Ablating layer: {top_3_layers}")

# Create multiple hooks
hooks = []
for layer_idx in top_3_layers:
    hook_fn = create_mlp_ablation_hook(layer_idx, clean_cache)
    hooks.append((f"blocks.{layer_idx}.mlp.hook_post", hook_fn))

# Run with multiple ablations
with model.hooks(fwd_hooks=hooks):
    multi_ablated_logits = model(tokens)

multi_ablated_probs = torch.softmax(multi_ablated_logits[0, -1, :], dim=-1)
multi_yes_prob = multi_ablated_probs[yes_token].item()
multi_no_prob = multi_ablated_probs[no_token].item()
multi_logit_diff = (multi_ablated_logits[0, -1, yes_token] - multi_ablated_logits[0, -1, no_token]).item()

print(f"Results with top 1 layer ablated:")
print(f"Yes: {baseline_yes_prob:.3f} → {multi_yes_prob:.3f} (Δ={multi_yes_prob-baseline_yes_prob:+.3f})")
print(f"No:  {baseline_no_prob:.3f} → {multi_no_prob:.3f} (Δ={multi_no_prob-baseline_no_prob:+.3f})")
print(f"Logit diff: {baseline_logit_diff:.3f} → {multi_logit_diff:.3f} (Δ={multi_logit_diff-baseline_logit_diff:+.3f})")

# Check if decision flipped
baseline_decision = "Yes" if baseline_yes_prob > baseline_no_prob else "No"
multi_ablated_decision = "Yes" if multi_yes_prob > multi_no_prob else "No"
print(f"\nDecision change: {baseline_decision} → {multi_ablated_decision}")
if baseline_decision != multi_ablated_decision:
    print("🔥 DECISION FLIPPED! These MLP layers are causally important.")
else:
    print("Decision unchanged, but probabilities shifted.")

Baseline - Yes: 0.181, No: 0.213
Baseline logit difference (Yes - No): -0.162

Testing ablation of MLP layers 13 to 27:
------------------------------------------------------------
Layer 13: Yes Δ=-0.026, No Δ=+0.154, LogitDiff Δ=-0.702
Layer 14: Yes Δ=-0.012, No Δ=+0.020, LogitDiff Δ=-0.163
Layer 15: Yes Δ=-0.032, No Δ=+0.170, LogitDiff Δ=-0.778
Layer 16: Yes Δ=+0.155, No Δ=+0.040, LogitDiff Δ=+0.446
Layer 17: Yes Δ=+0.057, No Δ=+0.036, LogitDiff Δ=+0.115
Layer 18: Yes Δ=+0.122, No Δ=+0.079, LogitDiff Δ=+0.199
Layer 19: Yes Δ=-0.132, No Δ=-0.138, LogitDiff Δ=-0.261
Layer 20: Yes Δ=-0.057, No Δ=-0.028, LogitDiff Δ=-0.236
Layer 21: Yes Δ=+0.176, No Δ=-0.186, LogitDiff Δ=+2.735
Layer 22: Yes Δ=-0.045, No Δ=+0.062, LogitDiff Δ=-0.541
Layer 23: Yes Δ=+0.067, No Δ=+0.076, LogitDiff Δ=+0.010
Layer 24: Yes Δ=+0.040, No Δ=-0.030, LogitDiff Δ=+0.351
Layer 25: Yes Δ=+0.108, No Δ=+0.038, LogitDiff Δ=+0.302
Layer 26: Yes Δ=-0.019, No Δ=-0.021, LogitDiff Δ=-0.006
Layer 27: Yes Δ=-0.097, No Δ=-0.146


Most impactful MLP layers (by absolute logit difference change):
------------------------------------------------------------
1. Layer 21: Logit diff change = +2.735
   Final probs after ablation - Yes: 0.357, No: 0.027
2. Layer 15: Logit diff change = -0.778
   Final probs after ablation - Yes: 0.150, No: 0.383
3. Layer 13: Logit diff change = -0.702
   Final probs after ablation - Yes: 0.155, No: 0.367
4. Layer 22: Logit diff change = -0.541
   Final probs after ablation - Yes: 0.136, No: 0.275
5. Layer 16: Logit diff change = +0.446
   Final probs after ablation - Yes: 0.336, No: 0.253
Ablating layer: [21]
Results with top 1 layer ablated:
Yes: 0.181 → 0.357 (Δ=+0.176)
No:  0.213 → 0.027 (Δ=-0.186)
Logit diff: -0.162 → 2.573 (Δ=+2.735)

Decision change: No → Yes
🔥 DECISION FLIPPED! These MLP layers are causally important.


In [20]:
prompt

'\nAnswer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?\n'

In [21]:
import torch
import numpy as np
from neel_plotly import *

# SPECIFY WHICH MLP LAYER TO ABLATE HERE
LAYER_TO_ABLATE = 21  # Change this to test different layers

# Your prompt and tokens (assuming already defined)
# prompt = "..."
# tokens = model.to_tokens(prompt)

# Get baseline (clean) run
print(f"Running baseline...")
clean_logits, clean_cache = model.run_with_cache(tokens)
yes_token = model.to_single_token("Yes")
no_token = model.to_single_token("No")

# Baseline probabilities across all layers
baseline_yes_probs = []
baseline_no_probs = []
layer_numbers = []

for layer in range(model.cfg.n_layers):
    hidden = clean_cache[f"blocks.{layer}.hook_resid_post"][0, -1, :]
    logits_layer = model.unembed(model.ln_final(hidden))
    probs_layer = torch.softmax(logits_layer, dim=-1)
    baseline_yes_probs.append(probs_layer[yes_token].item())
    baseline_no_probs.append(probs_layer[no_token].item())
    layer_numbers.append(layer)

print(f"Baseline final - Yes: {baseline_yes_probs[-1]:.3f}, No: {baseline_no_probs[-1]:.3f}")

# Create mean ablation hook for the specified MLP layer
def mlp_ablation_hook(mlp_out, hook):
    """Replace MLP output with its mean across sequence positions"""
    mean_activation = clean_cache[f"blocks.{LAYER_TO_ABLATE}.mlp.hook_post"].mean(dim=1, keepdim=True)
    mlp_out[:] = mean_activation
    return mlp_out

# Run with MLP layer ablated
print(f"Running with Layer {LAYER_TO_ABLATE} MLP ablated...")
with model.hooks(fwd_hooks=[(f"blocks.{LAYER_TO_ABLATE}.mlp.hook_post", mlp_ablation_hook)]):
    ablated_logits, ablated_cache = model.run_with_cache(tokens)

# Ablated probabilities across all layers
ablated_yes_probs = []
ablated_no_probs = []

for layer in range(model.cfg.n_layers):
    hidden = ablated_cache[f"blocks.{layer}.hook_resid_post"][0, -1, :]
    logits_layer = model.unembed(model.ln_final(hidden))
    probs_layer = torch.softmax(logits_layer, dim=-1)
    ablated_yes_probs.append(probs_layer[yes_token].item())
    ablated_no_probs.append(probs_layer[no_token].item())

print(f"Ablated final - Yes: {ablated_yes_probs[-1]:.3f}, No: {ablated_no_probs[-1]:.3f}")

# Calculate probability changes
yes_changes = [ablated - baseline for ablated, baseline in zip(ablated_yes_probs, baseline_yes_probs)]
no_changes = [ablated - baseline for ablated, baseline in zip(ablated_no_probs, baseline_no_probs)]

# Create comparison plots from layer 15 onwards
layer_15_start = 17
layer_numbers_focused = layer_numbers[layer_15_start:]
baseline_yes_focused = baseline_yes_probs[layer_15_start:]
baseline_no_focused = baseline_no_probs[layer_15_start:]
ablated_yes_focused = ablated_yes_probs[layer_15_start:]
ablated_no_focused = ablated_no_probs[layer_15_start:]
yes_changes_focused = yes_changes[layer_15_start:]
no_changes_focused = no_changes[layer_15_start:]

# 1. Baseline vs Ablated probabilities (Layer 15+)
baseline_data_focused = np.array([baseline_yes_focused, baseline_no_focused])
ablated_data_focused = np.array([ablated_yes_focused, ablated_no_focused])

# Create plots with vertical line indicating ablated layer
import plotly.graph_objects as go

def create_line_plot_with_vline(data, x_vals, title, line_labels, ablated_layer):
    """Create line plot with vertical line marking the ablated layer"""
    fig = go.Figure()
    
    # Add data lines
    for i, line_data in enumerate(data):
        fig.add_trace(go.Scatter(
            x=x_vals,
            y=line_data,
            mode='lines',
            name=line_labels[i],
            line=dict(width=2)
        ))
    
    # Add vertical line at ablated layer
    if ablated_layer in x_vals:
        fig.add_vline(
            x=ablated_layer, 
            line_dash="dash", 
            line_color="black", 
            line_width=2,
            annotation_text=f"Ablated L{ablated_layer}",
            annotation_position="top"
        )
    
    fig.update_layout(
        title=title,
        xaxis_title="Layer",
        yaxis_title="Probability",
        showlegend=True
    )
    
    fig.show()
    return fig

# 1. Baseline probabilities with ablation marker
create_line_plot_with_vline(
    baseline_data_focused,
    layer_numbers_focused,
    f"Baseline: Yes vs No Probabilities (Layer 17+) (€1,000, 330 mins)",
    ["Yes (Baseline)", "No (Baseline)"],
    LAYER_TO_ABLATE
)

# 2. Ablated probabilities with ablation marker
create_line_plot_with_vline(
    ablated_data_focused,
    layer_numbers_focused,
    f"Ablated L{LAYER_TO_ABLATE} MLP: Yes vs No Probabilities (Layer 17+) (€1,000, 330 mins)",
    ["Yes (Ablated)", "No (Ablated)"],
    LAYER_TO_ABLATE
)

# 3. Changes due to ablation with ablation marker
changes_data_focused = np.array([yes_changes_focused, no_changes_focused])
create_line_plot_with_vline(
    changes_data_focused,
    layer_numbers_focused,
    f"Probability Changes from Ablating L{LAYER_TO_ABLATE} MLP (Layer 17+) (€1,000, 330 mins)",
    ["Yes Change", "No Change"],
    LAYER_TO_ABLATE
)

# 4. Combined comparison with ablation marker
combined_data_focused = np.array([baseline_yes_focused, baseline_no_focused, ablated_yes_focused, ablated_no_focused])
create_line_plot_with_vline(
    combined_data_focused,
    layer_numbers_focused,
    f"Baseline vs Ablated L{LAYER_TO_ABLATE} MLP: All Probabilities (Layer 17+) (€1,000, 330 mins)",
    ["Yes (Baseline)", "No (Baseline)", "Yes (Ablated)", "No (Ablated)"],
    LAYER_TO_ABLATE
)

# Summary statistics
final_yes_change = yes_changes[-1]
final_no_change = no_changes[-1]
final_logit_diff_change = (ablated_logits[0, -1, yes_token] - ablated_logits[0, -1, no_token]).item() - baseline_logit_diff

print(f"\n" + "="*60)
print(f"ABLATION SUMMARY FOR LAYER {LAYER_TO_ABLATE} MLP:")
print(f"="*60)
print(f"Final Yes probability change: {final_yes_change:+.4f}")
print(f"Final No probability change:  {final_no_change:+.4f}")
print(f"Final logit diff change:      {final_logit_diff_change:+.4f}")

# Check decision flip
baseline_decision = "Yes" if baseline_yes_probs[-1] > baseline_no_probs[-1] else "No"
ablated_decision = "Yes" if ablated_yes_probs[-1] > ablated_no_probs[-1] else "No"
print(f"Decision: {baseline_decision} → {ablated_decision}")

if baseline_decision != ablated_decision:
    print(f"DECISION FLIPPED! Layer {LAYER_TO_ABLATE} MLP is causally important.")
else:
    print(f"Decision unchanged.")

# Show when the biggest change occurs
max_change_layer = np.argmax(np.abs(yes_changes))
print(f"Biggest Yes probability change occurs at layer {max_change_layer}: {yes_changes[max_change_layer]:+.4f}")

Running baseline...
Baseline final - Yes: 0.181, No: 0.213
Running with Layer 21 MLP ablated...


Ablated final - Yes: 0.357, No: 0.027



ABLATION SUMMARY FOR LAYER 21 MLP:
Final Yes probability change: +0.1764
Final No probability change:  -0.1857
Final logit diff change:      +2.7351
Decision: No → Yes
DECISION FLIPPED! Layer 21 MLP is causally important.
Biggest Yes probability change occurs at layer 27: +0.1764


# Further

In [22]:
# import torch
# import numpy as np
# from neel_plotly import *
# import plotly.graph_objects as go

# # LAYER 21 MLP INVESTIGATION
# LAYER = 21

# # 1. ACTIVATION PATCHING: Test different prompt variations
# print("="*60)
# print("1. ACTIVATION PATCHING WITH PROMPT VARIATIONS")
# print("="*60)

# # Original prompt
# original_prompt = """
# Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?
# """

# # Test variations to understand what Layer 21 responds to
# test_prompts = [
#     """
# Answer with one word. Do you accept compensation of 10.00 euros for waiting 330 minutes for food?
# """,
#     """
# Answer with one word. Do you reject compensation of 1000.00 euros for waiting 330 minutes for food?
# """
# ]

# def analyze_prompt_mlp_activation(prompt_text, layer_idx=LAYER):
#     """Analyze Layer 21 MLP activation for a given prompt"""
#     tokens = model.to_tokens(prompt_text)
#     logits, cache = model.run_with_cache(tokens)
    
#     # Get MLP activations (pre and post)
#     mlp_pre = cache[f"blocks.{layer_idx}.ln2.hook_normalized"][0, -1, :]   # Input to MLP (after LayerNorm)
#     mlp_post = cache[f"blocks.{layer_idx}.mlp.hook_post"][0, -1, :] # Output from MLP
    
#     # Get final decision
#     yes_token = model.to_single_token("Yes")
#     no_token = model.to_single_token("No")
#     final_probs = torch.softmax(logits[0, -1, :], dim=-1)
#     yes_prob = final_probs[yes_token].item()
#     no_prob = final_probs[no_token].item()
#     decision = "Yes" if yes_prob > no_prob else "No"
    
#     return {
#         'prompt': prompt_text[:60] + "..." if len(prompt_text) > 60 else prompt_text,
#         'mlp_pre': mlp_pre,
#         'mlp_post': mlp_post,
#         'yes_prob': yes_prob,
#         'no_prob': no_prob,
#         'decision': decision,
#         'cache': cache
#     }

# # Analyze original and variations
# original_result = analyze_prompt_mlp_activation(original_prompt)
# print(f"Original: {original_result['decision']} (Yes: {original_result['yes_prob']:.3f})")

# variation_results = []
# for i, prompt_var in enumerate(test_prompts):
#     result = analyze_prompt_mlp_activation(prompt_var)
#     variation_results.append(result)
#     print(f"Var {i+1}: {result['decision']} (Yes: {result['yes_prob']:.3f}) - {result['prompt']}")

# print("\n" + "="*60)
# print("2. MLP ACTIVATION SIMILARITY ANALYSIS")
# print("="*60)

# # Compare MLP activations between original and variations
# original_mlp = original_result['mlp_post']

# for i, var_result in enumerate(variation_results):
#     var_mlp = var_result['mlp_post']
    
#     # Cosine similarity
#     cosine_sim = torch.cosine_similarity(original_mlp, var_mlp, dim=0).item()
    
#     # L2 distance
#     l2_dist = torch.norm(original_mlp - var_mlp).item()
    
#     print(f"Var {i+1}: Cosine sim: {cosine_sim:.3f}, L2 dist: {l2_dist:.1f}, Decision: {var_result['decision']}")

# print("\n" + "="*60)
# print("3. NEURON-LEVEL ANALYSIS")
# print("="*60)

# # Get MLP neuron activations (after ReLU)
# mlp_acts = original_result['cache'][f"blocks.{LAYER}.mlp.hook_post"][0, -1, :]  # [d_model]

# # Find most active neurons
# top_k = 20
# top_neuron_vals, top_neuron_indices = torch.topk(torch.abs(mlp_acts), k=top_k)

# print(f"Top {top_k} most active neurons in Layer {LAYER} MLP:")
# for i, (idx, val) in enumerate(zip(top_neuron_indices, top_neuron_vals)):
#     print(f"{i+1:2d}. Neuron {idx:4d}: {mlp_acts[idx].item():+.3f}")

# print("\n" + "="*60)
# print("4. ABLATION OF TOP NEURONS")
# print("="*60)

# # Test ablating just the top few neurons
# def create_neuron_ablation_hook(layer_idx, neuron_indices):
#     """Ablate specific neurons in an MLP layer"""
#     def hook_fn(mlp_out, hook):
#         mlp_out[0, -1, neuron_indices] = 0  # Zero out specific neurons at last position
#         return mlp_out
#     return hook_fn

# # Test ablating top 5 most active neurons
# top_5_neurons = top_neuron_indices[:5]
# print(f"Ablating top 5 neurons: {top_5_neurons.tolist()}")

# neuron_hook = create_neuron_ablation_hook(LAYER, top_5_neurons)
# with model.hooks(fwd_hooks=[(f"blocks.{LAYER}.mlp.hook_post", neuron_hook)]):
#     neuron_ablated_logits = model(tokens)

# neuron_ablated_probs = torch.softmax(neuron_ablated_logits[0, -1, :], dim=-1)
# neuron_yes_prob = neuron_ablated_probs[yes_token].item()
# neuron_no_prob = neuron_ablated_probs[no_token].item()
# neuron_decision = "Yes" if neuron_yes_prob > neuron_no_prob else "No"

# print(f"After ablating top 5 neurons:")
# print(f"Yes: {original_result['yes_prob']:.3f} → {neuron_yes_prob:.3f}")
# print(f"No:  {original_result['no_prob']:.3f} → {neuron_no_prob:.3f}")
# print(f"Decision: {original_result['decision']} → {neuron_decision}")

# print("\n" + "="*60)
# print("5. FEATURE DIRECTION ANALYSIS")
# print("="*60)

# # Analyze what direction the MLP is pushing in embedding space
# mlp_output = original_result['mlp_post']  # [d_model]

# # Project onto Yes/No directions
# W_U = model.W_U
# yes_unembed = W_U[:, yes_token]
# no_unembed = W_U[:, no_token]

# yes_projection = torch.dot(mlp_output, yes_unembed).item()
# no_projection = torch.dot(mlp_output, no_unembed).item()

# print(f"MLP Layer {LAYER} output projection onto:")
# print(f"  Yes direction: {yes_projection:+.3f}")
# print(f"  No direction:  {no_projection:+.3f}")
# print(f"  Difference:    {yes_projection - no_projection:+.3f}")

# # Find top vocabulary items this MLP output is similar to
# mlp_logits = mlp_output @ W_U  # [vocab_size]
# top_vocab_vals, top_vocab_indices = torch.topk(mlp_logits, k=10)
# bottom_vocab_vals, bottom_vocab_indices = torch.topk(mlp_logits, k=10, largest=False)

# print(f"\nTop 10 tokens Layer {LAYER} MLP pushes TOWARD:")
# for i, (idx, val) in enumerate(zip(top_vocab_indices, top_vocab_vals)):
#     token_str = model.to_string(idx)
#     print(f"{i+1:2d}. '{token_str}': {val.item():+.3f}")

# print(f"\nTop 10 tokens Layer {LAYER} MLP pushes AWAY FROM:")
# for i, (idx, val) in enumerate(zip(bottom_vocab_indices, bottom_vocab_vals)):
#     token_str = model.to_string(idx)
#     print(f"{i+1:2d}. '{token_str}': {val.item():+.3f}")

# print("\n" + "="*60)
# print("6. CONTEXTUAL SENSITIVITY TEST")
# print("="*60)

# # Test how Layer 21 responds to different contexts
# context_prompts = [
#     "The weather is nice today. Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?",
#     "This is a fair deal. Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?", 
#     "This is highway robbery. Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?",
#     "Be reasonable. Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?",
# ]

# print("Testing contextual sensitivity:")
# for i, context_prompt in enumerate(context_prompts):
#     result = analyze_prompt_mlp_activation(context_prompt)
#     mlp_context = result['mlp_post']
#     context_sim = torch.cosine_similarity(original_mlp, mlp_context, dim=0).item()
#     print(f"Context {i+1}: {result['decision']} (Yes: {result['yes_prob']:.3f}), MLP sim: {context_sim:.3f}")

# print(f"\nINVESTIGATION COMPLETE FOR LAYER {LAYER} MLP")
# print("="*60)

In [23]:
# import torch
# import numpy as np
# from neel_plotly import *
# import plotly.graph_objects as go

# # LAYER 21 MLP INVESTIGATION
# LAYER = 21

# # 1. ACTIVATION PATCHING: Test different prompt variations
# print("="*60)
# print("1. ACTIVATION PATCHING WITH PROMPT VARIATIONS")
# print("="*60)

# # Original prompt
# original_prompt = """
# Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?
# """

# # Test variations to understand what Layer 21 responds to
# test_prompts = [
#     """
# Answer with one word. Do you accept compensation of 10.00 euros for waiting 330 minutes for food?
# """,
#     """
# Answer with one word. Do you reject compensation of 1000.00 euros for waiting 330 minutes for food?
# """
# ]

# def analyze_prompt_mlp_activation(prompt_text, layer_idx=LAYER):
#     """Analyze Layer 21 MLP activation for a given prompt"""
#     tokens = model.to_tokens(prompt_text)
#     logits, cache = model.run_with_cache(tokens)
    
#     # Get MLP activations (pre and post)
#     mlp_pre = cache[f"blocks.{layer_idx}.ln2.hook_normalized"][0, -1, :]   # Input to MLP (after LayerNorm)
#     # Get the final MLP output (after down-projection) - this should be d_model size
#     mlp_post = cache[f"blocks.{layer_idx}.hook_mlp_out"][0, -1, :] # Final MLP output back to d_model
    
#     # Get final decision
#     yes_token = model.to_single_token("Yes")
#     no_token = model.to_single_token("No")
#     final_probs = torch.softmax(logits[0, -1, :], dim=-1)
#     yes_prob = final_probs[yes_token].item()
#     no_prob = final_probs[no_token].item()
#     decision = "Yes" if yes_prob > no_prob else "No"
    
#     return {
#         'prompt': prompt_text,
#         'mlp_pre': mlp_pre,
#         'mlp_post': mlp_post,
#         'yes_prob': yes_prob,
#         'no_prob': no_prob,
#         'decision': decision,
#         'cache': cache,
#         'tokens': tokens,
#         'logits': logits
#     }

# # Analyze original and variations
# original_result = analyze_prompt_mlp_activation(original_prompt)
# print(f"Original: {original_result['decision']} (Yes: {original_result['yes_prob']:.3f})")

# variation_results = []
# for i, prompt_var in enumerate(test_prompts):
#     result = analyze_prompt_mlp_activation(prompt_var)
#     variation_results.append(result)
#     print(f"Var {i+1}: {result['decision']} (Yes: {result['yes_prob']:.3f}) - {result['prompt']}")

# print("\n" + "="*60)
# print("2. MLP ACTIVATION SIMILARITY ANALYSIS")
# print("="*60)

# # Compare MLP activations between original and variations
# original_mlp = original_result['mlp_post']

# for i, var_result in enumerate(variation_results):
#     var_mlp = var_result['mlp_post']
    
#     # Cosine similarity
#     cosine_sim = torch.cosine_similarity(original_mlp, var_mlp, dim=0).item()
    
#     # L2 distance
#     l2_dist = torch.norm(original_mlp - var_mlp).item()
    
#     print(f"Var {i+1}: Cosine sim: {cosine_sim:.3f}, L2 dist: {l2_dist:.1f}, Decision: {var_result['decision']}")

# print("\n" + "="*60)
# print("3. NEURON-LEVEL ANALYSIS")
# print("="*60)

# # Get MLP neuron activations (intermediate activations after ReLU, before down-projection)
# mlp_intermediate = original_result['cache'][f"blocks.{LAYER}.mlp.hook_post"][0, -1, :]  # [d_mlp]
# # Get final MLP output (after down-projection)
# mlp_acts = original_result['cache'][f"blocks.{LAYER}.hook_mlp_out"][0, -1, :]  # [d_model]

# # Find most active intermediate neurons (in the d_mlp space)
# top_k = 20
# top_neuron_vals, top_neuron_indices = torch.topk(torch.abs(mlp_intermediate), k=top_k)

# print(f"Top {top_k} most active intermediate neurons in Layer {LAYER} MLP:")
# for i, (idx, val) in enumerate(zip(top_neuron_indices, top_neuron_vals)):
#     print(f"{i+1:2d}. Neuron {idx:4d}: {mlp_intermediate[idx].item():+.3f}")

# print(f"\nFinal MLP output (d_model) statistics:")
# print(f"  Shape: {mlp_acts.shape}")
# print(f"  Mean: {mlp_acts.mean().item():+.3f}")
# print(f"  Std: {mlp_acts.std().item():+.3f}")
# print(f"  Max: {mlp_acts.max().item():+.3f}")
# print(f"  Min: {mlp_acts.min().item():+.3f}")

# print("\n" + "="*60)
# print("4. ABLATION OF TOP NEURONS")
# print("="*60)

# # Test ablating just the top few intermediate neurons
# def create_neuron_ablation_hook(layer_idx, neuron_indices):
#     """Ablate specific intermediate neurons in an MLP layer"""
#     def hook_fn(mlp_post, hook):
#         mlp_post[0, -1, neuron_indices] = 0  # Zero out specific neurons at last position
#         return mlp_post
#     return hook_fn

# # Test ablating top 5 most active intermediate neurons
# top_5_neurons = top_neuron_indices[:1]
# print(f"Ablating top 1 intermediate neurons: {top_5_neurons.tolist()}")

# tokens = original_result['tokens']
# yes_token = model.to_single_token("Yes")
# no_token = model.to_single_token("No")

# # Ablate the intermediate activations (after ReLU, before down-projection)
# neuron_hook = create_neuron_ablation_hook(LAYER, top_5_neurons)
# with model.hooks(fwd_hooks=[(f"blocks.{LAYER}.mlp.hook_post", neuron_hook)]):
#     neuron_ablated_logits = model(tokens)

# neuron_ablated_probs = torch.softmax(neuron_ablated_logits[0, -1, :], dim=-1)
# neuron_yes_prob = neuron_ablated_probs[yes_token].item()
# neuron_no_prob = neuron_ablated_probs[no_token].item()
# neuron_decision = "Yes" if neuron_yes_prob > neuron_no_prob else "No"

# print(f"After ablating top 5 intermediate neurons:")
# print(f"Yes: {original_result['yes_prob']:.3f} → {neuron_yes_prob:.3f}")
# print(f"No:  {original_result['no_prob']:.3f} → {neuron_no_prob:.3f}")
# print(f"Decision: {original_result['decision']} → {neuron_decision}")

# # Also test ablating the final MLP output directly
# print(f"\nTesting direct MLP output ablation:")
# def create_mlp_output_ablation_hook(layer_idx):
#     """Completely ablate the MLP output"""
#     def hook_fn(mlp_out, hook):
#         mlp_out[0, -1, :] = 0  # Zero out entire MLP output at last position
#         return mlp_out
#     return hook_fn

# mlp_output_hook = create_mlp_output_ablation_hook(LAYER)
# with model.hooks(fwd_hooks=[(f"blocks.{LAYER}.hook_mlp_out", mlp_output_hook)]):
#     mlp_ablated_logits = model(tokens)

# mlp_ablated_probs = torch.softmax(mlp_ablated_logits[0, -1, :], dim=-1)
# mlp_yes_prob = mlp_ablated_probs[yes_token].item()
# mlp_no_prob = mlp_ablated_probs[no_token].item()
# mlp_decision = "Yes" if mlp_yes_prob > mlp_no_prob else "No"

# print(f"After ablating entire MLP output:")
# print(f"Yes: {original_result['yes_prob']:.3f} → {mlp_yes_prob:.3f}")
# print(f"No:  {original_result['no_prob']:.3f} → {mlp_no_prob:.3f}")
# print(f"Decision: {original_result['decision']} → {mlp_decision}")

# print("\n" + "="*60)
# print("5. FEATURE DIRECTION ANALYSIS")
# print("="*60)

# # Analyze what direction the MLP is pushing in embedding space
# mlp_output = original_result['mlp_post']  # [d_model]

# # Get the unembedding matrix and check dimensions
# W_U = model.W_U  # Should be [d_model, vocab_size]
# print(f"MLP output shape: {mlp_output.shape}")
# print(f"W_U shape: {W_U.shape}")

# # Project onto Yes/No directions
# yes_unembed = W_U[:, yes_token]  # [d_model]
# no_unembed = W_U[:, no_token]    # [d_model]

# print(f"Yes unembed shape: {yes_unembed.shape}")
# print(f"No unembed shape: {no_unembed.shape}")

# # Fix the projection calculation
# yes_projection = torch.dot(mlp_output, yes_unembed).item()
# no_projection = torch.dot(mlp_output, no_unembed).item()

# print(f"MLP Layer {LAYER} output projection onto:")
# print(f"  Yes direction: {yes_projection:+.3f}")
# print(f"  No direction:  {no_projection:+.3f}")
# print(f"  Difference:    {yes_projection - no_projection:+.3f}")

# # Find top vocabulary items this MLP output is similar to
# mlp_logits = mlp_output @ W_U  # [vocab_size]
# top_vocab_vals, top_vocab_indices = torch.topk(mlp_logits, k=10)
# bottom_vocab_vals, bottom_vocab_indices = torch.topk(mlp_logits, k=10, largest=False)

# print(f"\nTop 10 tokens Layer {LAYER} MLP pushes TOWARD:")
# for i, (idx, val) in enumerate(zip(top_vocab_indices, top_vocab_vals)):
#     token_str = model.to_string(idx)
#     print(f"{i+1:2d}. '{token_str}': {val.item():+.3f}")

# print(f"\nTop 10 tokens Layer {LAYER} MLP pushes AWAY FROM:")
# for i, (idx, val) in enumerate(zip(bottom_vocab_indices, bottom_vocab_vals)):
#     token_str = model.to_string(idx)
#     print(f"{i+1:2d}. '{token_str}': {val.item():+.3f}")

# print("\n" + "="*60)
# print("6. CONTEXTUAL SENSITIVITY TEST")
# print("="*60)

# # Test how Layer 21 responds to different contexts
# context_prompts = [
#     "The weather is nice today. Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?",
#     "This is a fair deal. Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?", 
#     "This is highway robbery. Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?",
#     "Be reasonable. Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?",
# ]

# print("Testing contextual sensitivity:")
# for i, context_prompt in enumerate(context_prompts):
#     result = analyze_prompt_mlp_activation(context_prompt)
#     mlp_context = result['mlp_post']
#     context_sim = torch.cosine_similarity(original_mlp, mlp_context, dim=0).item()
#     print(f"Context {i+1}: {result['decision']} (Yes: {result['yes_prob']:.3f}), MLP sim: {context_sim:.3f}")

# print(f"\nINVESTIGATION COMPLETE FOR LAYER {LAYER} MLP")
# print("="*60)

# # Debug section to understand the dimension issue
# print("\n" + "="*60)
# print("DEBUG: DIMENSION ANALYSIS")
# print("="*60)

# print(f"Model config:")
# print(f"  d_model: {model.cfg.d_model}")
# print(f"  d_mlp: {model.cfg.d_mlp}")
# print(f"  vocab_size: {model.cfg.d_vocab}")

# print(f"\nActual tensor shapes:")
# print(f"  MLP output: {mlp_output.shape}")
# print(f"  W_U: {W_U.shape}")
# print(f"  Yes token ID: {yes_token}")
# print(f"  No token ID: {no_token}")

In [24]:
import torch
import numpy as np
from neel_plotly import *

# INVESTIGATE NEURON 8190 IN LAYER 21
LAYER = 21
NEURON = 8190

print("="*60)
print(f"INVESTIGATING NEURON {NEURON} IN LAYER {LAYER}")
print("="*60)

# First, let's check the model dimensions to ensure our neuron index is valid
print("MODEL DIMENSIONS:")
print(f"d_model: {model.cfg.d_model}")
print(f"d_mlp: {model.cfg.d_mlp}")
print(f"Layer {LAYER} MLP has {model.cfg.d_mlp} intermediate neurons (0 to {model.cfg.d_mlp-1})")
print(f"Investigating neuron {NEURON} {'✓' if NEURON < model.cfg.d_mlp else '✗ OUT OF BOUNDS!'}")
print()

if NEURON >= model.cfg.d_mlp:
    print(f"ERROR: Neuron {NEURON} is out of bounds. Max neuron index is {model.cfg.d_mlp-1}")
    print("Please choose a neuron index between 0 and", model.cfg.d_mlp-1)
else:
    # 1. ACTIVATION MAXIMIZATION - What inputs make this neuron fire most?
    print("1. TESTING DIFFERENT INPUTS TO MAXIMIZE NEURON ACTIVATION")
    print("-"*60)

    def get_neuron_activation(prompt_text, layer_idx=LAYER, neuron_idx=NEURON):
        """Get the activation of a specific intermediate neuron for a given prompt"""
        tokens = model.to_tokens(prompt_text)
        _, cache = model.run_with_cache(tokens)
        
        # Get the neuron's intermediate activation (after ReLU, before down-projection)
        mlp_intermediate_acts = cache[f"blocks.{layer_idx}.mlp.hook_post"][0, -1, :]  # [d_mlp]
        neuron_activation = mlp_intermediate_acts[neuron_idx].item()
        
        return neuron_activation, cache

    # Test various prompts to see what maximizes this neuron
    test_prompts = [
        # Original
        """
Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?
""",
        
        # Different amounts
        """
Answer with one word. Do you accept compensation of 50.00 euros for waiting 330 minutes for food?
""",
        """
Answer with one word. Do you accept compensation of 200.00 euros for waiting 330 minutes for food?
""",
        """
Answer with one word. Do you accept compensation of 1000.00 euros for waiting 330 minutes for food?
""",
        
        # Different times
        """
Answer with one word. Do you accept compensation of 100.00 euros for waiting 60 minutes for food?
""",
        """
Answer with one word. Do you accept compensation of 100.00 euros for waiting 600 minutes for food?
""",
        
        # Different framings
        """
Answer with one word. Do you reject compensation of 100.00 euros for waiting 330 minutes for food?
""",
        """
Answer with one word. Is 100.00 euros fair compensation for waiting 330 minutes?
""",
        
        # Different contexts
        """
Answer with one word. This is outrageous. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?
""",
        """
Answer with one word. This seems reasonable. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?
""",
        
        # Just numbers
        """
100.00 euros for 330 minutes
""",
        """
The amount is 100.00 euros
""",
        """
Wait time: 330 minutes
""",
        
        # Similar decision tasks  
        """
Answer with one word. Do you accept this job offer of 100.00 euros per day?
""",
        """
Answer with one word. Is this price of 100.00 euros acceptable?
"""
    ]

    neuron_activations = []
    for i, prompt in enumerate(test_prompts):
        activation, _ = get_neuron_activation(prompt.strip())
        short_prompt = prompt.strip()[:80] + "..." if len(prompt.strip()) > 80 else prompt.strip()
        neuron_activations.append((short_prompt, activation))
        print(f"{i+1:2d}. {activation:+.3f}: {short_prompt}")

    # Sort by activation strength
    neuron_activations.sort(key=lambda x: abs(x[1]), reverse=True)

    print(f"\nTOP 5 PROMPTS THAT MAXIMIZE NEURON {NEURON}:")
    print("-"*60)
    for i, (prompt, activation) in enumerate(neuron_activations[:5]):
        print(f"{i+1}. {activation:+.3f}: {prompt}")

    print("\n" + "="*60)
    print("2. NEURON ABLATION TEST")
    print("="*60)

    # Test what happens when we ablate JUST this neuron
    original_prompt = """
Answer with one word. Do you accept compensation of 100.00 euros for waiting 330 minutes for food?
"""

    # Get baseline
    tokens = model.to_tokens(original_prompt.strip())
    clean_logits, clean_cache = model.run_with_cache(tokens)

    yes_token = model.to_single_token("Yes")
    no_token = model.to_single_token("No")

    clean_probs = torch.softmax(clean_logits[0, -1, :], dim=-1)
    baseline_yes = clean_probs[yes_token].item()
    baseline_no = clean_probs[no_token].item()
    baseline_decision = "Yes" if baseline_yes > baseline_no else "No"

    print(f"Baseline - Yes: {baseline_yes:.3f}, No: {baseline_no:.3f}, Decision: {baseline_decision}")
    print(f"Baseline neuron {NEURON} activation: {clean_cache[f'blocks.{LAYER}.mlp.hook_post'][0, -1, NEURON].item():+.3f}")

    # Ablate just this neuron (intermediate activation)
    def single_neuron_ablation_hook(mlp_intermediate, hook):
        mlp_intermediate[0, -1, NEURON] = 0  # Zero out just this intermediate neuron
        return mlp_intermediate

    with model.hooks(fwd_hooks=[(f"blocks.{LAYER}.mlp.hook_post", single_neuron_ablation_hook)]):
        ablated_logits = model(tokens)

    ablated_probs = torch.softmax(ablated_logits[0, -1, :], dim=-1)
    ablated_yes = ablated_probs[yes_token].item()
    ablated_no = ablated_probs[no_token].item()
    ablated_decision = "Yes" if ablated_yes > ablated_no else "No"

    print(f"Ablated  - Yes: {ablated_yes:.3f}, No: {ablated_no:.3f}, Decision: {ablated_decision}")
    print(f"Change   - Yes: {ablated_yes - baseline_yes:+.3f}, No: {ablated_no - baseline_no:+.3f}")

    if baseline_decision != ablated_decision:
        print(f"🔥 DECISION FLIPPED! Neuron {NEURON} alone is causally important!")
    else:
        print(f"Decision unchanged, but probabilities shifted.")

    print("\n" + "="*60)
    print("3. NEURON OUTPUT DIRECTION ANALYSIS")
    print("="*60)

    # Get the output weight vector for this neuron
    # In TransformerLens, the MLP weights are accessed as:
    # W_in: [d_model, d_mlp] - projects from residual to intermediate
    # W_out: [d_mlp, d_model] - projects from intermediate back to residual
    
    # Get the down-projection weight for this neuron
    neuron_output_vector = model.blocks[LAYER].mlp.W_out[NEURON, :]  # [d_model]

    print(f"Neuron {NEURON} output vector shape: {neuron_output_vector.shape}")

    # What does this neuron's output vector push toward in vocabulary space?
    W_U = model.W_U  # [d_model, vocab_size]
    neuron_vocab_effects = neuron_output_vector @ W_U  # [vocab_size]

    # Get top tokens this neuron pushes toward/away from
    top_vals, top_indices = torch.topk(neuron_vocab_effects, k=15)
    bottom_vals, bottom_indices = torch.topk(neuron_vocab_effects, k=15, largest=False)

    print(f"\nTop 15 tokens neuron {NEURON} pushes TOWARD:")
    for i, (idx, val) in enumerate(zip(top_indices, top_vals)):
        token_str = model.to_string(idx)
        print(f"{i+1:2d}. '{token_str}' ({idx}): {val.item():+.3f}")

    print(f"\nTop 15 tokens neuron {NEURON} pushes AWAY FROM:")
    for i, (idx, val) in enumerate(zip(bottom_indices, bottom_vals)):
        token_str = model.to_string(idx)
        print(f"{i+1:2d}. '{token_str}' ({idx}): {val.item():+.3f}")

    # Check specifically for Yes/No
    yes_effect = neuron_vocab_effects[yes_token].item()
    no_effect = neuron_vocab_effects[no_token].item()
    print(f"\nNeuron {NEURON} direct effects:")
    print(f"  On 'Yes' token: {yes_effect:+.3f}")
    print(f"  On 'No' token:  {no_effect:+.3f}")
    print(f"  Yes - No diff:  {yes_effect - no_effect:+.3f}")

    print("\n" + "="*60)
    print("4. ACTIVATION PATTERN ANALYSIS")
    print("="*60)

    # Test systematic variations to understand what pattern this neuron detects
    amount_tests = [10.00, 25.00, 50.00, 75.00, 100.00, 150.00, 200.00, 500.00, 1000.00]
    time_tests = [30, 60, 120, 180, 240, 330, 420, 600, 720]

    print("Testing different AMOUNTS (330 minutes fixed):")
    amount_activations = []
    for amount in amount_tests:
        test_prompt = f"Answer with one word. Do you accept compensation of {amount:.2f} euros for waiting 330 minutes for food?"
        activation, _ = get_neuron_activation(test_prompt)
        amount_activations.append(activation)
        print(f"€{amount:7.2f}: {activation:+.3f}")

    print(f"\nTesting different TIMES (100.00 euros fixed):")
    time_activations = []
    for time in time_tests:
        test_prompt = f"Answer with one word. Do you accept compensation of 100.00 euros for waiting {time} minutes for food?"
        activation, _ = get_neuron_activation(test_prompt)
        time_activations.append(activation)
        print(f"{time:3d} min: {activation:+.3f}")

    # Plot the patterns
    try:
        line(
            [amount_activations],
            x=amount_tests,
            title=f"Neuron {NEURON} Activation vs Amount (€)",
            xaxis="Amount (euros)",
            yaxis="Neuron Activation",
            line_labels=[f"Neuron {NEURON}"]
        )

        line(
            [time_activations],
            x=time_tests,
            title=f"Neuron {NEURON} Activation vs Wait Time",
            xaxis="Wait Time (minutes)", 
            yaxis="Neuron Activation",
            line_labels=[f"Neuron {NEURON}"]
        )

        # Calculate hourly rate for each combination to see if neuron tracks this
        hourly_rates = [(amount * 60 / 330) for amount in amount_tests]
        line(
            [amount_activations],
            x=hourly_rates,
            title=f"Neuron {NEURON} Activation vs Hourly Rate",
            xaxis="Hourly Rate (€/hour)",
            yaxis="Neuron Activation", 
            line_labels=[f"Neuron {NEURON}"]
        )
    except Exception as e:
        print(f"Plotting failed: {e}")
        print("Continuing without plots...")

    print("\n" + "="*60)
    print("5. ADDITIONAL ANALYSIS: NEURON STATISTICS")
    print("="*60)
    
    # Get activation statistics across all test prompts
    all_activations = [act for _, act in neuron_activations]
    print(f"Neuron {NEURON} activation statistics:")
    print(f"  Mean: {np.mean(all_activations):+.3f}")
    print(f"  Std:  {np.std(all_activations):+.3f}")
    print(f"  Min:  {np.min(all_activations):+.3f}")
    print(f"  Max:  {np.max(all_activations):+.3f}")
    
    # Check if neuron is generally active or sparse
    active_count = sum(1 for act in all_activations if abs(act) > 0.1)
    print(f"  Active prompts (|activation| > 0.1): {active_count}/{len(all_activations)}")

print(f"\nINVESTIGATION COMPLETE FOR NEURON {NEURON}")
print("="*60)

INVESTIGATING NEURON 8190 IN LAYER 21
MODEL DIMENSIONS:
d_model: 3072
d_mlp: 8192
Layer 21 MLP has 8192 intermediate neurons (0 to 8191)
Investigating neuron 8190 ✓

1. TESTING DIFFERENT INPUTS TO MAXIMIZE NEURON ACTIVATION
------------------------------------------------------------
 1. +0.034: Answer with one word. Do you accept compensation of 100.00 euros for waiting 330...
 2. +0.073: Answer with one word. Do you accept compensation of 50.00 euros for waiting 330 ...
 3. +0.065: Answer with one word. Do you accept compensation of 200.00 euros for waiting 330...
 4. +0.144: Answer with one word. Do you accept compensation of 1000.00 euros for waiting 33...
 5. +0.035: Answer with one word. Do you accept compensation of 100.00 euros for waiting 60 ...
 6. -0.146: Answer with one word. Do you accept compensation of 100.00 euros for waiting 600...
 7. -4.802: Answer with one word. Do you reject compensation of 100.00 euros for waiting 330...
 8. +0.040: Answer with one word. Is 100.00


5. ADDITIONAL ANALYSIS: NEURON STATISTICS
Neuron 8190 activation statistics:
  Mean: -0.863
  Std:  +1.872
  Min:  -5.918
  Max:  +0.144
  Active prompts (|activation| > 0.1): 6/15

INVESTIGATION COMPLETE FOR NEURON 8190
